# Feature Engineering

In [2]:
import pandas as pd
import numpy as np
import re
import csv
import os

In [3]:
# Load resume extraction data (path relative to project root)
data_path = os.path.join("..", "data", "resume_extraction.csv")
df = pd.read_csv(data_path, on_bad_lines="skip", encoding="utf-8", encoding_errors="replace", low_memory=False)

# Strip column names (handles trailing space in "skills ")
df.columns = df.columns.str.strip()

print("Data loaded successfully.")

Data loaded successfully.


In [4]:
# Initial exploration
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nData types:")
print(df.dtypes)
print("\nMissing values per column:")
print(df.isnull().sum())
print("\n--- Sample value distributions ---")
for col in ["Gender", "Education", "Job_status"]:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False).head(15))

Shape: (1048575, 10)

Columns: ['Gender', 'Education', 'Specialization', 'interests', 'skills', 'Yearly salary in pounds', 'Certifications', 'Job_status', 'Job_title', 'Highest Qualification']

Data types:
Gender                     object
Education                  object
Specialization             object
interests                  object
skills                     object
Yearly salary in pounds     int64
Certifications             object
Job_status                 object
Job_title                  object
Highest Qualification      object
dtype: object

Missing values per column:
Gender                     244226
Education                  244226
Specialization             244226
interests                  257108
skills                     249919
Yearly salary in pounds         0
Certifications             258688
Job_status                 258614
Job_title                  454328
Highest Qualification      759317
dtype: int64

--- Sample value distributions ---

Gender:
Gender
Male   

In [5]:
# --- Preprocessing ---

# Placeholder values to treat as missing (for text columns except Job_status)
PLACEHOLDERS = ["No", "NA", "NO", ""]

# Text columns where we map placeholders to NaN (excluding Job_status and Certifications for now)
text_cols_for_placeholder = ["Education", "Specialization", "interests", "skills", "Job_title", "Highest Qualification"]
for col in text_cols_for_placeholder:
    if col in df.columns:
        df[col] = df[col].replace(PLACEHOLDERS, np.nan)
        df[col] = df[col].replace("", np.nan)

# Certifications: "No" means no certification - we'll use this for has_certification feature
# Keep as-is; we'll create has_certification from it later

# Yearly salary: coerce to numeric, invalid -> NaN
salary_col = "Yearly salary in pounds"
df[salary_col] = pd.to_numeric(df[salary_col], errors="coerce")

# Text normalization: Education
education_map = {
    "b.e": "BE", "be": "BE", "b.tech": "B.Tech", "btech": "B.Tech",
    "b.sc": "B.Sc", "bsc": "B.Sc", "bca": "BCA", "b.com": "B.Com", "bcom": "B.Com",
    "ba": "BA", "mba": "MBA", "m.sc": "M.Sc", "msc": "M.Sc", "m.tech": "M.Tech", "mtech": "M.Tech",
    "m.com": "M.Com", "mcom": "M.Com", "bms": "BMS", "b.m.s": "BMS",
    "b.pharmacy": "B.Pharmacy", "bpharmacy": "B.Pharmacy", "diploma": "Diploma",
    "mca": "MCA", "b.allb": "BALLB", "ballb": "BALLB", "law hons": "Law Hons",
    "bachelor of management studies": "BMS", "bachelor of journalism": "Bachelor of Journalism",
}
df["Education"] = df["Education"].astype(str).str.strip().str.lower()
df["Education"] = df["Education"].replace(education_map)

# Specialization: lowercase, strip
df["Specialization"] = df["Specialization"].astype(str).str.strip().str.lower()
df["Specialization"] = df["Specialization"].replace(["nan", ""], np.nan)

# Gender: standardize
df["Gender"] = df["Gender"].astype(str).str.strip()
df["Gender"] = df["Gender"].replace(["nan", ""], np.nan)

print("Preprocessing complete.")

Preprocessing complete.


In [6]:
# --- Skills and interests parsing ---

def parse_list_field(val):
    """Split on both ; and ,, strip whitespace, filter empty. Handles NaN."""
    if pd.isna(val) or (isinstance(val, str) and val.strip() in ["", "NO", "No", "NA"]):
        return []
    s = str(val).strip()
    # Replace semicolons with commas for unified splitting
    s = s.replace(";", ",")
    parts = [p.strip() for p in s.split(",") if p.strip()]
    return list(dict.fromkeys(parts))  # deduplicate while preserving order

df["skills_list"] = df["skills"].apply(parse_list_field)
df["interests_list"] = df["interests"].apply(parse_list_field)

print("Sample parsed skills:", df["skills_list"].iloc[0][:5], "...")
print("Sample parsed interests:", df["interests_list"].iloc[2][:3], "...")

Sample parsed skills: ['Critical Thinking', 'Analytic Thinking', 'SQL', 'Programming', 'Work under Pressure'] ...
Sample parsed interests: ['Sales/Marketing', 'Trading', 'Understand human behaviour'] ...


In [7]:
# --- Feature creation ---

# 3.1 Categorical features
cert_col = "Certifications"
df["has_certification"] = (
    (df[cert_col].notna()) & 
    (~df[cert_col].astype(str).str.strip().str.lower().isin(["no", "nan", ""]))
).astype(int)

df["is_employed"] = (df["Job_status"].astype(str).str.strip().str.lower() == "yes").astype(int)

# education_level: ordinal (Diploma=1, Bachelors=2, Masters=3, PhD=4)
education_level_map = {
    "Diploma": 1, "BCA": 2, "BA": 2, "B.Com": 2, "B.Sc": 2, "BE": 2, "B.Tech": 2,
    "BMS": 2, "B.Pharmacy": 2, "BALLB": 2, "Law Hons": 2, "Bachelor of Journalism": 2,
    "MBA": 3, "M.Sc": 3, "M.Tech": 3, "M.Com": 3, "MCA": 3,
}
df["education_level"] = df["Education"].map(education_level_map).fillna(2)  # default Bachelors

# specialization_domain: STEM, Business, Humanities, Other
stem_keywords = ["computer", "engineering", "mechanical", "electrical", "electronics", "civil", 
                 "technology", "math", "physics", "chemistry", "biology", "biotech", "ai", "data"]
business_keywords = ["commerce", "marketing", "management", "account", "finance", "mba", "sales"]
humanities_keywords = ["english", "psychology", "history", "political", "law", "journalism"]

def map_domain(spec):
    if pd.isna(spec) or spec == "nan": return "Other"
    s = str(spec).lower()
    if any(k in s for k in stem_keywords): return "STEM"
    if any(k in s for k in business_keywords): return "Business"
    if any(k in s for k in humanities_keywords): return "Humanities"
    return "Other"

df["specialization_domain"] = df["Specialization"].apply(map_domain)

# 3.2 Skills-based features
df["skill_count"] = df["skills_list"].apply(len)
df["interest_count"] = df["interests_list"].apply(len)

PROGRAMMING_SKILLS = ["python", "java", "sql", "c++", "javascript", "r", "c", "php", "matlab", "cpp"]
SOFT_SKILLS = ["communication", "leadership", "critical thinking", "problem solving", "team work", "teamwork"]

def has_skill(skills_list, keywords):
    skills_lower = [s.lower() for s in skills_list]
    return any(any(kw in s for kw in keywords) for s in skills_lower)

def count_skills_in_category(skills_list, keywords):
    return sum(1 for s in skills_list if any(kw in s.lower() for kw in keywords))

df["has_programming_skills"] = df["skills_list"].apply(lambda x: has_skill(x, PROGRAMMING_SKILLS)).astype(int)
df["has_soft_skills"] = df["skills_list"].apply(lambda x: has_skill(x, SOFT_SKILLS)).astype(int)
df["tech_skill_count"] = df["skills_list"].apply(lambda x: count_skills_in_category(x, PROGRAMMING_SKILLS))
df["soft_skill_count"] = df["skills_list"].apply(lambda x: count_skills_in_category(x, SOFT_SKILLS))

# 3.3 Education-job alignment
tech_job_keywords = ["software", "developer", "engineer", "data", "analyst", "programmer", "it", "tech"]
def education_job_match(row):
    domain = str(row["specialization_domain"]).lower()
    title = str(row["Job_title"]).lower() if pd.notna(row["Job_title"]) else ""
    if domain == "stem" and any(k in title for k in tech_job_keywords): return 1
    if domain == "business" and any(k in title for k in ["manager", "sales", "finance", "marketing"]): return 1
    return 0
df["education_job_match"] = df.apply(education_job_match, axis=1)

# highest_qualification_level
qual_map = {"bachelor": 1, "bachelors": 1, "b.sc": 1, "b.tech": 1, "be": 1,
            "master": 2, "masters": 2, "m.sc": 2, "m.tech": 2, "mba": 2, "mca": 2,
            "phd": 3, "doctorate": 3}
def qual_level(val):
    if pd.isna(val): return 1
    s = str(val).lower()
    for k, v in qual_map.items():
        if k in s: return v
    return 1
df["highest_qualification_level"] = df["Highest Qualification"].apply(qual_level)

# 3.4 Numeric features
df["yearly_salary"] = df["Yearly salary in pounds"]
salary_buckets = pd.cut(df["yearly_salary"], bins=[0, 50, 70, 200], labels=["low", "medium", "high"])
df["salary_bucket"] = salary_buckets.astype(str).replace("nan", "unknown")

# 3.5 Job title features
df["job_title_length"] = df["Job_title"].fillna("").astype(str).str.len()
df["job_title_clean"] = df["Job_title"].fillna("").astype(str).str.strip().str.lower()

print("Feature creation complete.")

Feature creation complete.


In [8]:
# --- Gender bias considerations and final feature matrix ---
# For screening model: EXCLUDE Gender from model input - use only as protected attribute for bias evaluation.
# Focus on job-relevant features: skills, education, certifications, experience.

# Feature columns for model input (job-relevant only; exclude Gender)
MODEL_FEATURES = [
    "education_level", "specialization_domain", "has_certification", "is_employed",
    "skill_count", "interest_count", "has_programming_skills", "has_soft_skills",
    "tech_skill_count", "soft_skill_count", "education_job_match", "highest_qualification_level",
    "yearly_salary", "salary_bucket", "job_title_length"
]

# Build feature matrix: include Gender only for bias analysis (e.g., fairness metrics)
df_features = df[MODEL_FEATURES + ["Gender"]].copy()

# One-hot encode categorical features for modeling (optional - keep raw for now; model can encode)
# salary_bucket and specialization_domain are categorical
df_features["salary_bucket"] = df_features["salary_bucket"].fillna("unknown")
df_features["specialization_domain"] = df_features["specialization_domain"].fillna("Other")

# Drop rows with too many missing values in key numeric features (optional)
# For now, keep all rows; fill NaN in yearly_salary with median for modeling
salary_median = df_features["yearly_salary"].median()
df_features["yearly_salary"] = df_features["yearly_salary"].fillna(salary_median)

print("Final feature matrix shape:", df_features.shape)
print("Columns:", list(df_features.columns))
print("\nGender distribution (for bias analysis):")
print(df_features["Gender"].value_counts(dropna=False))

Final feature matrix shape: (1048575, 16)
Columns: ['education_level', 'specialization_domain', 'has_certification', 'is_employed', 'skill_count', 'interest_count', 'has_programming_skills', 'has_soft_skills', 'tech_skill_count', 'soft_skill_count', 'education_job_match', 'highest_qualification_level', 'yearly_salary', 'salary_bucket', 'job_title_length', 'Gender']

Gender distribution (for bias analysis):
Gender
Male                 583717
NaN                  244226
Female               220398
Prefer not to say       234
Name: count, dtype: int64


In [ ]:
output_path = os.path.join("..", "data", "resume_features.csv")
df_features.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

print("\n--- Validation ---")
print("Duplicate rows:", df_features.duplicated().sum())
print("Row count:", len(df_features))

print("\n--- Categorical feature value counts ---")
for col in ["specialization_domain", "salary_bucket"]:
    print(f"\n{col}:")
    print(df_features[col].value_counts(dropna=False).head(10))

print("\n--- Numeric feature summary ---")
numeric_cols = ["education_level", "skill_count", "interest_count", "yearly_salary", "job_title_length"]
print(df_features[numeric_cols].describe())

Saved to ..\data\resume_features.csv

--- Validation ---
Duplicate rows: 998544
Row count: 1048575

--- Categorical feature value counts ---

specialization_domain:
specialization_domain
STEM          581548
Other         407437
Business       33446
Humanities     26144
Name: count, dtype: int64

salary_bucket:
salary_bucket
medium    511570
low       281247
high      255758
Name: count, dtype: int64

--- Numeric feature summary ---
       education_level   skill_count  interest_count  yearly_salary  \
count     1.048575e+06  1.048575e+06    1.048575e+06   1.048575e+06   
mean      2.000176e+00  4.866855e+00    2.400565e+00   5.999866e+01   
std       2.106866e-01  4.501140e+00    2.824577e+00   1.183180e+01   
min       1.000000e+00  0.000000e+00    0.000000e+00   4.000000e+01   
25%       2.000000e+00  1.000000e+00    1.000000e+00   5.000000e+01   
50%       2.000000e+00  5.000000e+00    2.000000e+00   6.000000e+01   
75%       2.000000e+00  7.000000e+00    3.000000e+00   7.000000e+0